Groundwater | Flow Modeling Track

# Step 6: Model Validation — Can the Model Predict What It Hasn’t Seen?

> **The 10 steps at a glance:** 1-Goal → 2-Perceptual → 3-MODFLOW → 4-Build → 5-Calibrate → **6-Validate** → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**Story so far:** In Step 5 you calibrated the Limmat Valley MODFLOW 6 model using PEST++ with ~20 pilot points, 4 real AWEL wells, 5 synthetic observations, and a pumping-test prior. The calibrated model fits the observations reasonably well — but does fitting calibration data mean the model can *predict* heads at locations it hasn’t seen? That’s the question validation answers.

| **Core content** | ~15 minutes |
|:--|:--|
| **Optional: LOO cross-validation & advanced topics** | +45 minutes |

---
## Learning Objectives

By the end of this notebook, you will be able to:

1. **Distinguish** between model verification, calibration, and validation
2. **Explain** why a steady-state model cannot use a temporal train/test split
3. **Implement** leave-one-out cross-validation for spatial hold-out testing *(optional)*
4. **Interpret** LOO prediction errors and compare them to calibration fit *(optional)*
5. **Assess** whether a model is fit for its intended purpose
6. **Describe** alternative validation approaches for different modelling contexts *(optional)*

---

## Prerequisites

Before starting this notebook, you should have:
- **Completed [05f_calibration.ipynb](05f_calibration.ipynb)** — the calibration workspace must exist
- A calibrated MODFLOW 6 model with spatially varying K from PEST++
- Familiarity with calibration metrics (RMSE, ME, R²) from Step 5

In [ ]:
# Setup
import sys
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

import flopy
from flopy.mf6 import MFSimulation

from IPython.display import display, HTML

# Add support modules to path
sys.path.append(os.path.abspath('../../_SUPPORT/src'))
sys.path.append(os.path.abspath('../../_SUPPORT/src/scripts'))
sys.path.append(os.path.abspath('../../_SUPPORT/src/scripts/scripts_exercises'))

from data_utils import (
    download_named_file,
    get_default_data_folder,
)
from model_io_utils import load_base_simulation, format_budget_summary
from plot_utils import quick_model_plot

# Checkpoint utilities
from shared_functions import check_task_with_solution, create_multiple_choice

import calibration_utils as cu
from setup_pest_calibration import (
    build_pest_setup,
    run_pestpp_glm,
    get_calibrated_k_field,
    apply_calibrated_parameters,
    _interpolate_pp_to_grid,
    _nearest_pilot_point,
    run_loo_cross_validation,
)

# Plotting defaults
plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['font.size'] = 12

---

## Introduction

Three terms are often confused but have distinct meanings:

| Step | Question | Methods |
|------|----------|---------|
| **Verification** | Is the model solving the equations correctly? | Mass balance, analytical benchmarks |
| **Calibration** | Can the model match the data we used to tune it? | RMSE, R², residual analysis |
| **Validation** | Can the model predict data it has not seen? | Hold-out tests, cross-validation, independent data |

Calibration tests **fit**; validation tests **prediction**. A model that fits its calibration data perfectly can still fail to predict at new locations — this is **overfitting**.

### The Steady-State Challenge

For transient models, validation is straightforward: reserve a portion of the time series for calibration and another one for validation (e.g., calibrate on 2010–2015, validate on 2016–2020). For our steady-state model, this temporal split is impossible: we calibrated to long-term mean head values. There is no time dimension to split. Available alternatives for steady-state validation:
1. Spatial hold-out / leave-one-out (LOO) cross-validation
2. Independent data types (flux measurements, tracer tests)
3. Physical plausibility checks

We’ll use all three, with **LOO cross-validation** as the main approach.

In [ ]:
# Checkpoint 1: Why can't we do a temporal train/test split?
create_multiple_choice('task06_checkpoint_1')

---

## 1 — Load Calibrated Model and Observations

In [ ]:
# --- 1. Define paths ---
print("--- 1. Define paths ---")
DATA_DIR = get_default_data_folder()

sim_name = 'limmat_valley'
nb4_workspace = os.path.join(DATA_DIR, 'notebook4_model')
calib_workspace = os.path.join(DATA_DIR, 'calibration')
val_workspace = os.path.join(DATA_DIR, 'validation')

# Check that calibration workspace exists (produced by NB5)
if not os.path.exists(os.path.join(calib_workspace, 'mfsim.nam')):
    raise FileNotFoundError(
        f"Calibration workspace not found at {calib_workspace}\n"
        "Please run Notebook 5 first to generate the calibrated model."
    )

# Create fresh validation workspace
if os.path.exists(val_workspace):
    shutil.rmtree(val_workspace)
os.makedirs(val_workspace)
print(f"Validation workspace: {val_workspace}")

# --- 2. Load NB4 simulation for reference grid/idomain ---
print("--- 2. Load NB4 simulation for reference grid/idomain ---")
sim = load_base_simulation(nb4_workspace)
gwf = sim.get_model(sim_name)

modelgrid = gwf.modelgrid
modelgrid.set_coord_info(crs="EPSG:2056")

disv = gwf.get_package('DISV')
idomain = disv.idomain.array

# Load boundary GeoDataFrame
boundary_path = download_named_file(name='model_boundary', data_type='gis')
boundary_gdf = gpd.read_file(boundary_path)

print(f"Grid type: {modelgrid.grid_type}")
print(f"Active cells: {(idomain.flatten() > 0).sum()}")

# --- 3. Load observations (reproduce NB5 exactly) ---
print("--- 3. Load observations (reproduce NB5 exactly) ---")
# Real AWEL observations
obs_gdf = cu.load_awel_observations(DATA_DIR, boundary_gdf, use_mean=True)
print(f"Real AWEL wells: {len(obs_gdf)}")

# Reproduce synthetic observations with same seed as NB5
top = disv.top.array.flatten()
botm = disv.botm.array.flatten()

k_reference, k_conditioned, cond_info = cu.generate_conditioned_k_field(
    sim, gwf, modelgrid, idomain, top, botm,
    obs_gdf=obs_gdf, seed=42, noise_std=0.25,
    variogram_range=3000.0, anisotropy_angle=-30.0, anisotropy_scaling=3.0,
)

# Run with reference K, then restore K=20
gwf.npf.k.set_data(k_reference)
sim.write_simulation()
success, _ = sim.run_simulation(silent=True)
if not success:
    raise RuntimeError("Reference model run failed.")
head_ref = gwf.output.head().get_data()
gwf.npf.k.set_data(200.0)

# River cells for exclusion
riv = gwf.get_package('RIV')
riv_data = riv.stress_period_data.get_data(0) if riv is not None else None
riv_cells = np.array([r[1] for r in riv_data]) if riv_data is not None else np.array([])

river_gis_path = os.path.join(os.path.dirname(boundary_path), 'AV_Gewasser_-OGD.gpkg')
river_all = gpd.read_file(river_gis_path)
river_gdf = river_all[
    river_all['GEWAESSERNAME'].isin(['Limmat', 'Sihl'])
    & river_all.intersects(boundary_gdf.geometry.iloc[0])
]

synth_gdf = cu.generate_synthetic_observations(
    modelgrid, head_ref, idomain,
    n_points=5, noise_std=0.5, seed=42,
    exclude_cells=riv_cells, river_gdf=river_gdf,
    boundary_polygon=boundary_gdf.geometry.iloc[0],
    avoid_boundaries_m=100, exclude_north_of_river=True,
)

obs_gdf = cu.combine_observations(obs_gdf, synth_gdf)
print(cu.get_observation_summary(obs_gdf))

# --- 4. Apply calibrated K field and Sihl leakance multiplier ---
print("--- 4. Apply calibrated K field and Sihl leakance multiplier ---")
pest_ws = os.path.join(calib_workspace, 'pest_setup')

cal_result = apply_calibrated_parameters(
    gwf, pest_ws, modelgrid, river_gdf, boundary_gdf,
    variogram_range=600.0, anisotropy_angle=-30.0, anisotropy_scaling=3.0,
    set_npf_flags=False,
)
k_field_cal = cal_result['k_field']
sihl_cell_ids = cal_result['sihl_cell_ids']

# Cell centres needed for LOO setup (Section 5)
xc = modelgrid.xcellcenters
yc = modelgrid.ycellcenters

# Enable specific discharge output for flow-direction plotting
gwf.npf.save_specific_discharge.set_data(True)

# Run calibrated model
sim.write_simulation()
success_cal, _ = sim.run_simulation(silent=True)
if not success_cal:
    raise RuntimeError("Calibrated model run failed.")
head_cal = gwf.output.head().get_data()
print("Calibrated model run successful.")

# --- 5. Compute baseline calibration metrics ---
print("--- 5. Compute baseline calibration metrics ---")
sim_heads_cal = cu.extract_heads_at_observations(head_cal, obs_gdf, modelgrid)
obs_gdf['sim_head_calibrated'] = sim_heads_cal

valid_mask = ~np.isnan(sim_heads_cal)
obs_valid = obs_gdf[valid_mask].copy()

# Metrics at ALL observations
metrics_all = cu.calculate_calibration_metrics(
    obs_valid['head_m'].values, obs_valid['sim_head_calibrated'].values,
)
print("Calibration metrics (all 9 observations):")
print(cu.format_metrics_table(metrics_all))

# Metrics at REAL wells only (this is the full-calibration reference)
real_obs = obs_valid[~obs_valid['is_synthetic']].copy()
metrics_real = cu.calculate_calibration_metrics(
    real_obs['head_m'].values, real_obs['sim_head_calibrated'].values,
)
cal_rmse_real = metrics_real['RMSE']
print(f"\nCalibration RMSE at real wells only: {cal_rmse_real:.3f} m")
print(f"(This is the baseline we'll compare LOO-RMSE against.)")

# --- 6. Calibration scatter plot and residual map ---
print("--- 6. Calibration scatter plot and residual map ---")
fig = cu.plot_calibration_scatter(
    obs_valid['head_m'].values,
    obs_valid['sim_head_calibrated'].values,
    obs_gdf=obs_valid,
    title='Full Calibration (all 9 obs)',
)
plt.show()

residuals = obs_valid['sim_head_calibrated'].values - obs_valid['head_m'].values
fig = cu.plot_residual_map(
    obs_valid, residuals,
    boundary_gdf=boundary_gdf,
    title='Residual Map \u2014 Full Calibration',
)
plt.show()

print("This is calibration quality, now let's test prediction quality.")

In [ ]:
# Checkpoint 2: Full-calibration RMSE at the 4 real AWEL wells
check_task_with_solution('task06_checkpoint_2')

---

## 2 — Model Verification

Verification asks: does the model solve the equations correctly? This is a necessary but not sufficient condition for a good model. A model that fails verification is useless; a model that passes verification is not necessarily validated. Indeed, verification confirms the implementation is correct, but says nothing about whether the model’s *predictions* are accurate.

Mass balance verification: in our case, for the mass balance, verification pass is expected (Water balance error < 1%), since MODFLOW’s iterative solver ensures convergence. 

In [ ]:
# --- Water balance ---
budget_df = format_budget_summary(gwf, sim)
display(budget_df)

pct_error = budget_df.loc['PERCENT ERROR', 'Net (m3/d)']
if isinstance(pct_error, (int, float)) and pct_error < 1.0:
    print("\nWater balance error < 1% \u2014 PASS")
else:
    print(f"\nWater balance error = {pct_error} \u2014 review model setup")

In [ ]:
# Checkpoint 3: Water balance error (%)
check_task_with_solution('task06_checkpoint_3')

---

## 3 — Physical Plausibility

Even good calibration metrics can come from a physically unreasonable model. In this section, we check whether the calibrated K field and head distribution make physical sense.

In [ ]:
# --- K-field plausibility ---
print("--- K-field plausibility ---")
k_flat = k_field_cal.flatten()
k_active = k_flat[idomain.flatten() > 0]

print("Calibrated K field statistics:")
print(f"  Min:  {k_active.min():.1f} m/d")
print(f"  Max:  {k_active.max():.1f} m/d")
print(f"  Mean: {k_active.mean():.1f} m/d")
print(f"  Median: {np.median(k_active):.1f} m/d")
print(f"\nComparison K values:")
print(f"  Pumping test K:         ~300 m/d")
print(f"  Literature (Limmat gravel): 1–300 m/d")

if k_active.min() >= 1.0 and k_active.max() <= 300.0:
    print("\n  K values within plausible range for Limmat Valley gravels.")
else:
    print("\n  Some K values outside expected range — check calibration.")

# Quick K field map
fig, ax = plt.subplots(figsize=(12, 8))
k_plot = np.where(idomain.flatten() > 0, k_flat, np.nan)
quick_model_plot(modelgrid, k_plot, boundary_gdf=boundary_gdf,
                 cmap='viridis', colorbar_label='K [m/d]',
                 title='Calibrated Hydraulic Conductivity', ax=ax)
fig.tight_layout()
plt.show()

# --- Head distribution plausibility ---
print("--- Head distribution plausibility ---")
heads_flat = head_cal.flatten() if head_cal.ndim > 1 else head_cal
heads_plot = np.where(idomain.flatten() > 0, heads_flat, np.nan)
valid_heads = heads_flat[idomain.flatten() > 0]

print("Calibrated head distribution statistics:")
print(f"  Range: {np.nanmin(valid_heads):.2f} \u2013 {np.nanmax(valid_heads):.2f} m a.s.l.")
print(f"  Mean:  {np.nanmean(valid_heads):.2f} m a.s.l.")

fig, ax = plt.subplots(figsize=(12, 8))

# Use FloPy's PlotMapView for native DISV support
pmv = flopy.plot.PlotMapView(modelgrid=modelgrid, ax=ax)
pc = pmv.plot_array(heads_plot, cmap='viridis')
plt.colorbar(pc, ax=ax, label='Head (m a.s.l.)', shrink=0.7)

# Head isolines (tri_mask=True removes triangles between non-neighbouring cells)
n_contours = 12
levels = np.linspace(np.nanmin(valid_heads), np.nanmax(valid_heads), n_contours)
cs = pmv.contour_array(heads_plot, levels=levels, tri_mask=True,
                       colors='black', linewidths=0.8, alpha=0.7)
ax.clabel(cs, inline=True, fontsize=8, fmt='%.0f')

# Clip contour lines to model boundary polygon
# (FloPy's tricontour extends beyond the irregular domain via Delaunay convex hull)
from matplotlib.path import Path as MplPath
from matplotlib.patches import PathPatch

boundary_geom = boundary_gdf.geometry.iloc[0]
if boundary_geom.geom_type == 'MultiPolygon':
    boundary_geom = max(boundary_geom.geoms, key=lambda g: g.area)
boundary_coords = np.array(boundary_geom.exterior.coords)
clip_path = MplPath(boundary_coords)
clip_patch = PathPatch(clip_path, transform=ax.transData, facecolor='none', edgecolor='none')
ax.add_patch(clip_patch)
# matplotlib >=3.8 removed .collections; set clip on all contour artists
if hasattr(cs, 'collections'):
    for coll in cs.collections:
        coll.set_clip_path(clip_patch)
else:
    cs.set_clip_path(clip_patch)

# Boundary outline
boundary_gdf.boundary.plot(ax=ax, color='black', linewidth=1.5)

# Overlay observation wells
real_obs_plot = obs_gdf[~obs_gdf['is_synthetic']]
synth_obs_plot = obs_gdf[obs_gdf['is_synthetic']]
if len(real_obs_plot) > 0:
    ax.scatter(real_obs_plot.geometry.x, real_obs_plot.geometry.y,
               s=100, c='red', edgecolors='black', linewidth=1, zorder=5,
               label='AWEL wells')
if len(synth_obs_plot) > 0:
    ax.scatter(synth_obs_plot.geometry.x, synth_obs_plot.geometry.y,
               s=60, c='#E67E22', marker='^', edgecolors='black', linewidth=0.8, zorder=5,
               label='Synthetic wells')
ax.legend(loc='lower left')
ax.set_title('Calibrated Head Distribution')
ax.set_aspect('equal')
fig.tight_layout()
plt.show()



# --- Summary of plausibility checks ---
print("--- Physical plausibility summary: ---")
print(f"  K range:        {k_active.min():.1f}\u2013{k_active.max():.1f} m/d "
      f"(expected: 1\u2013300 m/d)")
print(f"  Head range:     {np.nanmin(valid_heads):.1f}\u2013{np.nanmax(valid_heads):.1f} m a.s.l. "
      f"(expected: 380\u2013430 m a.s.l.)")

print("All plausibility checks passed: the model is internally consistent.")
print("But this doesn't tell us whether predictions at unobserved locations")
print("are reliable. That requires validation.")

In [ ]:
# Checkpoint 4: Does passing verification + plausibility = validated?
create_multiple_choice('task06_checkpoint_4')

---

## 4 — Leave-One-Out Cross-Validation Theory (*Optional*) 

<details>
<summary><strong>
View the optionnal section
</summary></strong>

### Concept

Leave-one-out (LOO) cross-validation is the most rigorous spatial validation available when you have very few observation wells. For our 4 real AWEL wells, we run **4 folds**:

| Fold | Held-out well | Calibration wells | Synthetic obs |
|------|--------------|-------------------|---------------|
| 1 | Well A | Wells B, C, D | All 5 |
| 2 | Well B | Wells A, C, D | All 5 |
| 3 | Well C | Wells A, B, D | All 5 |
| 4 | Well D | Wells A, B, C | All 5 |

In each fold:
1. **Drop** one real well from the observation set
2. **Recalibrate** with PEST++ using the remaining observations
3. **Predict** the head at the held-out well
4. Record the **prediction error**: $e_i = h_{\text{predicted}} - h_{\text{observed}}$

### LOO-RMSE

$$\text{LOO-RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^{N} e_i^2}$$

**Interpretation:**
- LOO-RMSE $\approx$ calibration RMSE $\rightarrow$ model generalises well
- LOO-RMSE $\gg$ calibration RMSE $\rightarrow$ possible overfitting

### Why do synthetic observations stay in all folds?

With only **3 real wells** remaining per fold, the 20-pilot-point calibration would be wildly underdetermined without the 5 synthetic observations providing spatial coverage. The synthetic points act as spatial constraints that prevent physically unreasonable K fields.

The LOO test is **exclusively on real data** — we only compute prediction errors at held-out real wells.


</details>

In [ ]:
# Checkpoint 5: Should synthetic observations be held out?
create_multiple_choice('task06_checkpoint_5')

---

## 5 — Run LOO Cross-Validation (*Optional*) 

This runs 4 PEST++ calibrations (one per fold). Expect **4–8 minutes** total runtime.

In [ ]:
# --- Prepare LOO parameters (mirror NB5 exactly) ---
K_pumping_test = 300.0  # from pumping test analysis in NB5

# Pumping test location: median of active domain
active_mask = idomain.flatten() > 0
active_x = xc[active_mask]
active_y = yc[active_mask]
pt_location = (float(np.median(active_x)), float(np.median(active_y)))

# Real well IDs for LOO
real_ids = obs_gdf[~obs_gdf['is_synthetic']]['obs_id'].tolist()
print(f"Real wells for LOO: {real_ids}")
print(f"Pumping test location: ({pt_location[0]:.0f}, {pt_location[1]:.0f})")
print(f"K from pumping test: {K_pumping_test:.1f} m/d")

# --- Run LOO cross-validation ---
loo_results = run_loo_cross_validation(
    nb4_workspace=nb4_workspace,
    val_workspace=val_workspace,
    sim_name=sim_name,
    modelgrid=modelgrid,
    obs_gdf=obs_gdf,
    real_well_ids=real_ids,
    K_pumping_test=K_pumping_test,
    pt_location=pt_location,
    idomain=idomain,
    sihl_cell_ids=sihl_cell_ids,
    n_pilot_points=20,
    k_initial=200.0,
    k_lower=10.0,
    k_upper=600.0,
    pt_weight=1.5,
    reg_weight=0.5,
    variogram_range=600.0,
    anisotropy_angle=-30.0,
    anisotropy_scaling=3.0,
    verbose=True,
)

# --- Display results table ---
loo_df = pd.DataFrame([
    {
        'Fold': r['fold'] + 1,
        'Held-out Well': r['held_out_well'],
        'Observed (m)': f"{r['obs_head']:.2f}",
        'Predicted (m)': f"{r['predicted_head']:.2f}" if not np.isnan(r['predicted_head']) else 'FAILED',
        'Error (m)': f"{r['prediction_error']:+.2f}" if not np.isnan(r['prediction_error']) else 'N/A',
        'Train RMSE (m)': f"{r['calib_rmse']:.2f}" if not np.isnan(r['calib_rmse']) else 'N/A',
        'PEST OK': r['pest_success'],
    }
    for r in loo_results
])

print("LOO Cross-Validation Results")
print("=" * 80)
display(loo_df)

# --- Compute LOO-RMSE and compare to full calibration ---
valid_errors = [r['prediction_error'] for r in loo_results
                if not np.isnan(r['prediction_error'])]

if len(valid_errors) > 0:
    loo_rmse = np.sqrt(np.mean(np.array(valid_errors)**2))
    loo_me = np.mean(valid_errors)
    loo_mae = np.mean(np.abs(valid_errors))

    print("LOO Cross-Validation Summary")
    print("=" * 50)
    print(f"  Successful folds:    {len(valid_errors)} / {len(loo_results)}")
    print(f"  LOO-RMSE:            {loo_rmse:.3f} m")
    print(f"  LOO Mean Error:      {loo_me:+.3f} m")
    print(f"  LOO Mean Abs Error:  {loo_mae:.3f} m")
    print(f"")
    print(f"  Full-cal RMSE (real): {cal_rmse_real:.3f} m")
    print(f"  LOO / Cal ratio:     {loo_rmse / cal_rmse_real:.2f}")
    print()
    if loo_rmse / cal_rmse_real < 2.0:
        print("  LOO-RMSE is within 2x of calibration RMSE \u2014 reasonable generalisation.")
    else:
        print("  LOO-RMSE is >2x calibration RMSE \u2014 possible overfitting.")
else:
    print("No valid LOO results \u2014 all folds failed.")
    loo_rmse = np.nan

In [ ]:
# Checkpoint 6: LOO-RMSE (m)
check_task_with_solution('task06_checkpoint_6')

---

## 6 — Interpret LOO Results (*Optionnal*)

<details>
<summary><strong>
View the optionnal section
</summary></strong>

**LOO-RMSE vs calibration RMSE:**
- If the ratio is close to 1, the model generalises well: removing one well doesn’t dramatically change the prediction at that location.
- If the ratio is much greater than 1, the model may be overfitting: it adjusts K to fit specific wells at the expense of spatial generalisability.

**Spatial limitation:** All 4 real wells are clustered in the eastern part of the domain. LOO only validates predictions *within this cluster*, not across the full domain. The model’s predictive capability in the central or western aquifer remains untested.

**Connection to equifinality:** In NB5 we saw that different K–recharge combinations can produce similar head fits. LOO reveals whether the calibrated solution is robust: if removing one well drastically changes the K field, this signals that the calibration is poorly constrained.

**What would improve confidence:** More observation wells with better spatial distribution, independent flux data (river baseflow), and transient calibration.

</details>

In [ ]:
# --- 4-panel figure: K field per fold with held-out well ---
folds_with_k = [r for r in loo_results if r['k_field'] is not None]

if len(folds_with_k) > 0:
    n_panels = len(folds_with_k)
    fig, axes = plt.subplots(1, n_panels, figsize=(5 * n_panels, 7))
    if n_panels == 1:
        axes = [axes]

    for ax, r in zip(axes, folds_with_k):
        k_plot = np.where(idomain.flatten() > 0, r['k_field'].flatten(), np.nan)
        quick_model_plot(modelgrid, k_plot, boundary_gdf=boundary_gdf,
                         cmap='viridis', colorbar_label='K [m/d]',
                         title=f"Fold {r['fold']+1}: drop {r['held_out_well']}", ax=ax)

        # Mark held-out well as star
        held_out = obs_gdf[obs_gdf['obs_id'] == r['held_out_well']].iloc[0]
        ax.scatter(held_out.geometry.x, held_out.geometry.y,
                   s=300, c='red', marker='*', edgecolors='black',
                   linewidth=1.5, zorder=6)
        if not np.isnan(r['prediction_error']):
            ax.annotate(f"e={r['prediction_error']:+.1f} m",
                        (held_out.geometry.x, held_out.geometry.y),
                        xytext=(8, -15), textcoords='offset points',
                        fontsize=9, fontweight='bold', color='red',
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    fig.suptitle('LOO Cross-Validation: Calibrated K per Fold', fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()
else:
    print("No K fields available for plotting.")

In [ ]:
# --- LOO scatter plot: predicted vs observed ---
valid_results = [r for r in loo_results if not np.isnan(r['predicted_head'])]

if len(valid_results) > 0:
    obs_vals = np.array([r['obs_head'] for r in valid_results])
    pred_vals = np.array([r['predicted_head'] for r in valid_results])

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(obs_vals, pred_vals, c='red', s=150, edgecolors='black',
               linewidth=1.5, zorder=3)

    # 1:1 line
    all_vals = np.concatenate([obs_vals, pred_vals])
    margin = (all_vals.max() - all_vals.min()) * 0.1
    lims = [all_vals.min() - margin, all_vals.max() + margin]
    ax.plot(lims, lims, 'k--', linewidth=1, label='1:1 line')
    ax.set_xlim(lims)
    ax.set_ylim(lims)

    # Annotate with well IDs and errors
    for r in valid_results:
        ax.annotate(f"{r['held_out_well']}\n({r['prediction_error']:+.1f} m)",
                    (r['obs_head'], r['predicted_head']),
                    xytext=(10, 5), textcoords='offset points', fontsize=9)

    ax.set_xlabel('Observed Head (m a.s.l.)')
    ax.set_ylabel('LOO Predicted Head (m a.s.l.)')
    ax.set_title(f'LOO Cross-Validation (n={len(valid_results)}, '
                 f'LOO-RMSE={loo_rmse:.2f} m)')
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print("No valid LOO predictions to plot.")

In [ ]:
# Checkpoint 7: If LOO-RMSE >> calibration RMSE, what does this suggest?
create_multiple_choice('task06_checkpoint_7')

---

## 7 — Other Validation Approaches (*Optional*)

<details>
<summary><strong>
View the optionnal section
</summary></strong>

### 7.1 - Alternatives for steady-state models

| Approach | Requirement | Our situation |
|----------|-------------|---------------|
| **Spatial splitting** | Enough wells for cal/val groups | 4 wells — too few |
| **LOO cross-validation** | N≥3 real wells | Our approach (N=4) |
| **Independent data types** | Flux obs, tracers, thermal profiles | Not available |
| **Predictive uncertainty** | PEST++ IES, null-space Monte Carlo | Advanced (NB7) |
| **Formal model comparison** | Multiple conceptual models (AIC, BIC) | Beyond scope |

### 7.2 - How transient models are validated

If we had transient head data, validation would be much more powerful:

- **Temporal split:** Calibrate on 2010–2015, validate on 2016–2020. Each time step at each well gives an independent validation point — with 4 wells and 5 years of monthly data, that's ~240 validation points instead of our 4.
- **Hindcasting:** Calibrate on recent data, predict historical conditions.
- **Data assimilation:** Continuously update the model as new observations arrive.

The key advantage: time series give **statistical power** that spatial-only data lacks.

### 7.3 - Professional Communication — Model Status

| Status | Appropriate uses | Not appropriate for |
|--------|-----------------|---------------------|
| **Unvalidated** | Screening, conceptual understanding | Design, regulatory |
| **Partially validated** (our case) | Preliminary design, scenario comparison | Final permits |
| **Fully validated** | Regulatory submissions, engineering design | — |

Our model is **partially validated**: we have LOO results at 4 clustered wells, plausibility checks pass, and verification confirms the numerics. To achieve full validation, we would need:
- 10+ wells with good spatial coverage
- Independent flux or gradient data
- Ideally a transient extension with temporal split testing

### 7.4 - Model Confidence Classification

Professional guidelines provide frameworks for classifying model confidence, helping modellers communicate how much trust to place in their results.

**Australian Groundwater Modelling Guidelines (Barnett et al., 2012) — Confidence Level Classification:**

| Class | Calibration | Validation | Data availability | Appropriate use |
|---|---|---|---|---|
| **Class 1** | Poor or none | None | Limited | Conceptual understanding only |
| **Class 2** | Moderate | Limited | Moderate | Scenario comparison, planning |
| **Class 3** | Good quantitative | Comprehensive | Extensive | Regulatory, engineering design |

**FH-DGGV Leitfaden (2024) — Model categories:**

| Category (German) | English equivalent | Typical use |
|---|---|---|
| **Prinzipmodell** | Principle model | Process understanding, screening |
| **Planungsmodell** | Planning model | Design support, scenario analysis |
| **Bewirtschaftungsmodell** | Management model | Operational decisions, regulatory compliance |

> **🤔 Think about it:**
> Where does our Limmat Valley model sit in these classifications? Consider: we have 4 real observation wells, a single-layer steady-state setup, and LOO cross-validation at a spatially clustered set of wells. Is this a Class 1, 2, or 3 model? A Prinzipmodell or a Planungsmodell?

</details>

---

## Summary

**What we did:**
- Loaded the NB5-calibrated model and reproduced identical observations
- **Verified** the model (water balance < 1%)
- Checked **physical plausibility** (K, heads, and gradients in reasonable ranges)
- Ran **leave-one-out cross-validation** (4 folds, 4 PEST++ calibrations)
- Compared LOO-RMSE to calibration RMSE to assess generalisation

**Key findings:**
- Verification and plausibility checks pass (necessary but not sufficient)
- LOO-RMSE provides the first quantitative measure of **predictive capability**
- All 4 real wells are spatially clustered \u2014 LOO validates only within the cluster

**Limitations:**
- Only 4 LOO folds \u2014 low statistical power
- No validation in the central or western domain
- No independent flux data to cross-check

---

## What Youre Taking Forward

| From this notebook | Used in |
|-------------------|---------|
| LOO-RMSE as predictive uncertainty estimate | NB7 (sensitivity/uncertainty) |
| Calibrated + validated model | NB8 (scenario application) |
| Awareness of spatial data gaps | NB9 (documentation) |
| Model status classification | NB10 (communication) |

---

## Next Steps

**Continue to:** [07f_sensitivity_uncertainty.ipynb](07f_sensitivity_uncertainty.ipynb) \u2014 where we explore how parameter uncertainty propagates to predictions.

After that: [08f_model_application.ipynb](08f_model_application.ipynb) \u2014 where we apply the calibrated model to real management questions.

---

# References

- Anderson, M.P., Woessner, W.W., Hunt, R.J. (2015). *Applied Groundwater Modeling* (2nd ed.). Academic Press.
- Barnett, B. et al. (2012). *Australian Groundwater Modelling Guidelines*. National Water Commission. [PDF](https://www.epa.govt.nz/assets/FileAPI/proposal/NSP000028/Hearings-Week-01/c2cd52352e/02-Australian-Groundwater-modelling-guidelines.pdf)
- FH-DGGV (2024). *Leitfaden Modellkalibrierung und -prognose (KalPro)*. Fachsektion Hydrogeologie in der DGGV.
- Hill, M.C., Tiedeman, C.R. (2007). *Effective Groundwater Model Calibration*. Wiley.
- Refsgaard, J.C., Henriksen, H.J. (2004). Modelling guidelines — terminology and guiding principles. *Advances in Water Resources*, 27(1), 71–82.